# 02 — Loading Real Documents (TXT, Markdown, PDF, Web & Metadata)

**By the end of this notebook, you can ingest plain text, Markdown, PDFs, web pages (with offline fallback), and international Unicode safely.**

### Learning Objectives
- Understand LangChain's `Document(page_content, metadata)` data structure.
- Load `.txt`, `.md`, `.pdf`, and HTML content consistently with proper UTF-8 encoding.
- Attach normalized metadata fields (`source`, `doc_type`, `page`) for later filtering and citations.

### Prerequisites
- Notebooks 00–01

## 1. Setup
Let's connect this notebook to our shared project helpers (`utils/models.py`), load `.env` (defaulting to free offline `local` mode), and initialize our LLM and embeddings.

In [1]:
# !pip install -q -r ../requirements.txt  # Uncomment if running in a fresh remote environment
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'utils').exists() else Path.cwd().parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
from utils.models import get_llm, get_embeddings, resolve_project_root
project_root = resolve_project_root()
data_dir = project_root / 'data'
llm = get_llm()
embeddings = get_embeddings()
print(f'Ready! Root={project_root.name} | LLM={type(llm).__name__}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready! Root=rag-from-scratch-course | LLM=LocalGroundedChatModel


## 2. Concept: Universal Adapter Pattern for Documents

Think of **Document Loaders** like universal travel power adapters: whether your company knowledge lives inside plain `.txt` files, formatted `.md` release notes, multi-page `.pdf` briefs, or live web pages, every loader plugs them into one standard output: a Python list of **Document** objects.

Every **Document** holds exactly two attributes:
- `.page_content`: The extracted clean text string.
- `.metadata`: A Python `dict` (like a library catalog card) storing `source`, `page`, `doc_type`, or permissions.

```
company_handbook.txt ---> [TextLoader] ---\
  recent_changelog.md  ---> [TextLoader] ----+--> List[Document(page_content=..., metadata={...})]
  policy_brief.pdf     ---> [PyPDFLoader] --/
  https://... / HTML   ---> [Web / Fallback]/
```

## 3. Step-by-Step Build

### Step 1: Load Text and Markdown Files with Normalized Metadata
Let's load `.txt` and `.md` files and enrich their `metadata` dictionaries so filenames stay concise rather than exposing messy system paths.

In [2]:
from langchain_community.document_loaders import TextLoader
files = [("company_handbook.txt", "handbook"), ("recent_changelog.md", "changelog")]
all_docs = []
for fname, dtype in files:
    for d in TextLoader(str(data_dir / fname), encoding="utf-8").load():
        d.metadata.update({"source": fname, "doc_type": dtype})
        all_docs.append(d)
print(f"Loaded {len(all_docs)} text/markdown docs. Meta[1]:", all_docs[1].metadata)
        

Loaded 2 text/markdown docs. Meta[1]: {'source': 'recent_changelog.md', 'doc_type': 'changelog'}


C:\Users\Rahat\AppData\Local\Temp\ipykernel_6852\2037937166.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


> **What just happened?**
> Clean metadata (`source='recent_changelog.md'`) ensures our citations remain portable across laptops and OSes.

### Step 2: Extract Text from a Real PDF Document
PDFs contain binary layout commands rather than raw text. Let's inspect `policy_brief.pdf` using `PyPDFLoader` (or `pypdf`) and preview its extracted page text.

In [3]:
from langchain_community.document_loaders import PyPDFLoader
pdf_docs = PyPDFLoader(str(data_dir / "policy_brief.pdf")).load()
for d in pdf_docs:
    d.metadata["source"] = "policy_brief.pdf"
    d.metadata["doc_type"] = "safety"
print(f"PDF Pages: {len(pdf_docs)} | Chars: {len(pdf_docs[0].page_content)}")
print("Preview:", pdf_docs[0].page_content[:120].replace("\n", " "))
        

PDF Pages: 1 | Chars: 1147
Preview: Novasphere Dynamics - Autonomous Robotics Safety & Deployment Brief Document ID: SAF-2026-B | Access Level: Internal & E


> **What just happened?**
> `PyPDFLoader` automatically splits multi-page PDFs into one `Document` per page with `page` numbers in metadata.

### Step 3: Web Loading with Safe Offline Fallback
Fetching live URLs can fail if a learner is offline or a site blocks bots. Always wrap web loaders with graceful error handling and fallback parsing.

In [4]:
from bs4 import BeautifulSoup
from langchain_core.documents import Document
sample_html = "<html><body><h1>Novasphere Docs</h1><p>gRPC v2 runs on port 50051.</p></body></html>"
soup = BeautifulSoup(sample_html, "html.parser")
web_doc = Document(page_content=soup.get_text(" ", strip=True), metadata={"source": "docs.html", "doc_type": "web"})
print("Extracted HTML text:", web_doc.page_content)
        

Extracted HTML text: Novasphere Docs gRPC v2 runs on port 50051.


> **What just happened?**
> Stripping HTML boilerplate with `BeautifulSoup` keeps noisy `<div>` and `<script>` tags out of your vector index.

### Step 4: Handle International Unicode & Filenames with Spaces Safely
Real enterprise documents contain non-ASCII languages (e.g., Bangla, Japanese, accented European letters) and spaces in filenames. Let's test `pathlib.Path` + `utf-8` handling.

In [5]:
import tempfile
with tempfile.TemporaryDirectory() as tmp:
    p = Path(tmp) / "my policy doc (2026).txt"
    p.write_text("নিরাপত্তা নীতিমালা — Zurich Hub allows dogs on Tuesdays.", encoding="utf-8")
    unicode_doc = TextLoader(str(p), encoding="utf-8").load()[0]
print("Loaded safely:", unicode_doc.page_content)
        

Loaded safely: নিরাপত্তা নীতিমালা — Zurich Hub allows dogs on Tuesdays.


> **What just happened?**
> Always specifying `encoding='utf-8'` and using `Path` objects avoids Windows CP-1252 crashes and broken paths with spaces.

## 4. Try It Yourself
Experiment by editing the starter ideas below before peeking at the solution!

1. **Use shared helper** — *Hint:* Call `load_corpus_documents(data_dir)` from `utils.helpers` and check total loaded docs.
2. **Count characters per doc_type** — *Hint:* Loop through loaded docs and print filename + total word count.

<details>
<summary><strong>Click to reveal sample solution</strong></summary>

Run the cell below to verify your solution.
</details>

In [6]:
from utils.helpers import load_corpus_documents
corpus = load_corpus_documents(data_dir)
for d in corpus:
    words = len(d.page_content.split())
    print(f"- {d.metadata['source']:<22} type={d.metadata['doc_type']:<10} words={words}")
        

- company_handbook.txt   type=handbook   words=411
- product_faq.md         type=faq        words=293
- recent_changelog.md    type=changelog  words=170
- policy_brief.pdf       type=safety     words=163


## 5. Common Mistakes
- Loading raw scanned-image PDFs without OCR — `PyPDFLoader` reads embedded text; image-only scans extract empty strings unless OCR is added.
- Leaving absolute paths (`C:\Users\...`) inside `.metadata['source']` — breaks relative source filtering and leaks usernames.
- Forgetting error handling on `WebBaseLoader` requests when URLs return 404 or timeout.

## 6. Recap
- Every LangChain Document combines `.page_content` (str) and `.metadata` (dict).
- `TextLoader(..., encoding='utf-8')`, `PyPDFLoader`, and `BeautifulSoup` cover 90% of corporate files safely.
- Clean, consistent metadata laid here powers filtering, RBAC security, and citations in later notebooks.

## 7. Next Notebook
Next up (`03_splitting_text.ipynb`): Learn why naive slicing cuts words in half and how Recursive & Markdown splitters preserve complete thoughts!

## 8. Video Chapter Outline
1. **What Is a Document Object?** — Inspecting page_content vs metadata.
2. **Loading TXT & Markdown Files** — Clean paths and UTF-8 handling.
3. **Reading PDFs Safely** — Page-by-page extraction with PyPDFLoader.
4. **HTML Scraping & Fallbacks** — Extracting readable prose without HTML noise.
5. **Edge Cases: Unicode & Spaces** — Preventing OS path and encoding bugs.